In [44]:
from pyspark.sql import SparkSession  #main entry poin to spark

spark = SparkSession.builder \
    .appName("GoToSpark-Test") \
    .getOrCreate()

print(spark.version)

4.2.0


In [45]:
df_taxi = spark.read.csv(
    "/workspace/data/nyctaxi.csv",
    header = True,
    inferSchema = True
)
display(df_taxi)  

DataFrame[Pickup_DateTime: string, DropOff_datetime: string, PUlocationID: int, DOlocationID: string, SR_Flag: string, Dispatching_base_number: string, Dispatching_base_num: string]

In [46]:
df_location = spark.read.csv(
    "/workspace/data/nyclocations.csv",
    header = True,
    inferSchema = True
)

display(df_location)

DataFrame[PUlocationID: int, location: string, street: string, zip_code: int]

In [4]:
df_taxi.count()

87829

In [5]:
df_location.count()

300

In [6]:
df_join = df_taxi.join(df_location)

In [7]:
df_join.show()

+---------------+----------------+------------+------------+-------+-----------------------+--------------------+------------+---------------+----------------+--------+
|Pickup_DateTime|DropOff_datetime|PUlocationID|DOlocationID|SR_Flag|Dispatching_base_number|Dispatching_base_num|PUlocationID|       location|          street|zip_code|
+---------------+----------------+------------+------------+-------+-----------------------+--------------------+------------+---------------+----------------+--------+
|3/14/2018 16:09| 3/14/2018 16:45|         241|          47|      1|                 B02836|                null|           1|      Riverdale|  Jamaica Avenue|   11432|
|3/14/2018 16:09| 3/14/2018 16:45|         241|          47|      1|                 B02836|                null|           2|       Brooklyn|  Madison Avenue|   10010|
|3/14/2018 16:09| 3/14/2018 16:45|         241|          47|      1|                 B02836|                null|           3|       Bushwick|Lexington Ave

In [8]:
df_join.count()

26348700

In [10]:
df_join.explain(True)

== Parsed Logical Plan ==
Join Inner
:- Relation [Pickup_DateTime#17,DropOff_datetime#18,PUlocationID#19,DOlocationID#20,SR_Flag#21,Dispatching_base_number#22,Dispatching_base_num#23] csv
+- Relation [PUlocationID#41,location#42,street#43,zip_code#44] csv

== Analyzed Logical Plan ==
Pickup_DateTime: string, DropOff_datetime: string, PUlocationID: int, DOlocationID: string, SR_Flag: string, Dispatching_base_number: string, Dispatching_base_num: string, PUlocationID: int, location: string, street: string, zip_code: int
Join Inner
:- Relation [Pickup_DateTime#17,DropOff_datetime#18,PUlocationID#19,DOlocationID#20,SR_Flag#21,Dispatching_base_number#22,Dispatching_base_num#23] csv
+- Relation [PUlocationID#41,location#42,street#43,zip_code#44] csv

== Optimized Logical Plan ==
Join Inner
:- Relation [Pickup_DateTime#17,DropOff_datetime#18,PUlocationID#19,DOlocationID#20,SR_Flag#21,Dispatching_base_number#22,Dispatching_base_num#23] csv
+- Relation [PUlocationID#41,location#42,street#43,zip

In [12]:
df_join2 = df_taxi.join(df_location,df_taxi.PUlocationID==df_location.PUlocationID)

In [13]:
df_join2.show()

+---------------+----------------+------------+------------+-------+-----------------------+--------------------+------------+------------------+------------------+--------+
|Pickup_DateTime|DropOff_datetime|PUlocationID|DOlocationID|SR_Flag|Dispatching_base_number|Dispatching_base_num|PUlocationID|          location|            street|zip_code|
+---------------+----------------+------------+------------+-------+-----------------------+--------------------+------------+------------------+------------------+--------+
|3/14/2018 16:09| 3/14/2018 16:45|         241|          47|      1|                 B02836|                null|         241|         Riverdale|    Jamaica Avenue|   11432|
|3/19/2018 13:48| 3/19/2018 13:54|          76|          76|      1|                 B02836|                null|          76|Bedford-Stuyvesant|   Flatbush Avenue|   11201|
|3/23/2018 16:30| 3/23/2018 17:24|          50|         223|      1|                 B02836|                null|          50|    

In [14]:
df_join2.explain(True)

== Parsed Logical Plan ==
Join Inner, (PUlocationID#19 = PUlocationID#41)
:- Relation [Pickup_DateTime#17,DropOff_datetime#18,PUlocationID#19,DOlocationID#20,SR_Flag#21,Dispatching_base_number#22,Dispatching_base_num#23] csv
+- Relation [PUlocationID#41,location#42,street#43,zip_code#44] csv

== Analyzed Logical Plan ==
Pickup_DateTime: string, DropOff_datetime: string, PUlocationID: int, DOlocationID: string, SR_Flag: string, Dispatching_base_number: string, Dispatching_base_num: string, PUlocationID: int, location: string, street: string, zip_code: int
Join Inner, (PUlocationID#19 = PUlocationID#41)
:- Relation [Pickup_DateTime#17,DropOff_datetime#18,PUlocationID#19,DOlocationID#20,SR_Flag#21,Dispatching_base_number#22,Dispatching_base_num#23] csv
+- Relation [PUlocationID#41,location#42,street#43,zip_code#44] csv

== Optimized Logical Plan ==
Join Inner, (PUlocationID#19 = PUlocationID#41)
:- Filter isnotnull(PUlocationID#19)
:  +- Relation [Pickup_DateTime#17,DropOff_datetime#18,PU

In [15]:
df_join2.count()

87829

In [16]:
df_join3 = df_taxi.join(df_location,df_taxi.PUlocationID==df_location.PUlocationID,"inner")

In [17]:
df_join3.count()

87829

In [20]:
df_leftjoin = df_taxi.join(df_location,df_taxi.PUlocationID==df_location.PUlocationID,"left")

In [21]:
df_leftjoin.count()

87829

In [22]:
df_leftjoin = df_location.join(df_taxi,df_taxi.PUlocationID==df_location.PUlocationID,"left")

In [23]:
df_leftjoin.count()

87874

In [24]:
df_leftanti = df_location.join(df_taxi,df_taxi.PUlocationID==df_location.PUlocationID,"leftanti")

In [25]:
df_leftanti.count()

45

In [47]:
from pyspark.sql.functions import broadcast

In [27]:
df_join3 = df_taxi.join(broadcast(df_location),df_taxi.PUlocationID==df_location.PUlocationID,"inner")

In [28]:
df_join3.explain(True)

== Parsed Logical Plan ==
Join Inner, (PUlocationID#19 = PUlocationID#41)
:- Relation [Pickup_DateTime#17,DropOff_datetime#18,PUlocationID#19,DOlocationID#20,SR_Flag#21,Dispatching_base_number#22,Dispatching_base_num#23] csv
+- ResolvedHint (strategy=broadcast)
   +- Relation [PUlocationID#41,location#42,street#43,zip_code#44] csv

== Analyzed Logical Plan ==
Pickup_DateTime: string, DropOff_datetime: string, PUlocationID: int, DOlocationID: string, SR_Flag: string, Dispatching_base_number: string, Dispatching_base_num: string, PUlocationID: int, location: string, street: string, zip_code: int
Join Inner, (PUlocationID#19 = PUlocationID#41)
:- Relation [Pickup_DateTime#17,DropOff_datetime#18,PUlocationID#19,DOlocationID#20,SR_Flag#21,Dispatching_base_number#22,Dispatching_base_num#23] csv
+- ResolvedHint (strategy=broadcast)
   +- Relation [PUlocationID#41,location#42,street#43,zip_code#44] csv

== Optimized Logical Plan ==
Join Inner, (PUlocationID#19 = PUlocationID#41), rightHint=(st

In [48]:
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

'-1'

In [32]:
10485760/1024/1024

10.0

In [34]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 100 * 1024 * 1024)

In [35]:
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

'104857600'

In [36]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [37]:
df_join3.explain(True)

== Parsed Logical Plan ==
Join Inner, (PUlocationID#19 = PUlocationID#41)
:- Relation [Pickup_DateTime#17,DropOff_datetime#18,PUlocationID#19,DOlocationID#20,SR_Flag#21,Dispatching_base_number#22,Dispatching_base_num#23] csv
+- ResolvedHint (strategy=broadcast)
   +- Relation [PUlocationID#41,location#42,street#43,zip_code#44] csv

== Analyzed Logical Plan ==
Pickup_DateTime: string, DropOff_datetime: string, PUlocationID: int, DOlocationID: string, SR_Flag: string, Dispatching_base_number: string, Dispatching_base_num: string, PUlocationID: int, location: string, street: string, zip_code: int
Join Inner, (PUlocationID#19 = PUlocationID#41)
:- Relation [Pickup_DateTime#17,DropOff_datetime#18,PUlocationID#19,DOlocationID#20,SR_Flag#21,Dispatching_base_number#22,Dispatching_base_num#23] csv
+- ResolvedHint (strategy=broadcast)
   +- Relation [PUlocationID#41,location#42,street#43,zip_code#44] csv

== Optimized Logical Plan ==
Join Inner, (PUlocationID#19 = PUlocationID#41), rightHint=(st

In [ ]:
2. Sort-Merge Join 
Shuffle both the datasets on the key column
Sort the data (any Partitions)
then it will merge the sorted datasets


In [38]:
spark.conf.get("spark.sql.join.preferSortMergeJoin")

'true'

In [40]:
spark.conf.set("spark.sql.join.preferSortMergeJoin", False)

In [ ]:
spark.conf.get("spark.sql.join.preferSortMergeJoin")

In [52]:
df_leftjoin = df_taxi.join(df_location.hint("merge"),df_taxi.PUlocationID==df_location.PUlocationID,"left")

In [53]:
df_leftjoin.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (9)
+- SortMergeJoin LeftOuter (8)
   :- Sort (3)
   :  +- Exchange (2)
   :     +- Scan csv  (1)
   +- Sort (7)
      +- Exchange (6)
         +- Filter (5)
            +- Scan csv  (4)


(1) Scan csv 
Output [7]: [Pickup_DateTime#311, DropOff_datetime#312, PUlocationID#313, DOlocationID#314, SR_Flag#315, Dispatching_base_number#316, Dispatching_base_num#317]
Batched: false
Location: InMemoryFileIndex [file:/workspace/data/nyctaxi.csv]
ReadSchema: struct<Pickup_DateTime:string,DropOff_datetime:string,PUlocationID:int,DOlocationID:string,SR_Flag:string,Dispatching_base_number:string,Dispatching_base_num:string>

(2) Exchange
Input [7]: [Pickup_DateTime#311, DropOff_datetime#312, PUlocationID#313, DOlocationID#314, SR_Flag#315, Dispatching_base_number#316, Dispatching_base_num#317]
Arguments: hashpartitioning(PUlocationID#313, 200), ENSURE_REQUIREMENTS, [plan_id=1076]

(3) Sort
Input [7]: [Pickup_DateTime#311, DropOff_datetime#312, PUlocationID#313,